# 2023_01_02

## using optuna
data와 사용하는cotrol variable, hparams와 같이 params가 다 바뀌고 셋팅을 해주어야 하는 상황임   
이를 직접하기에는 무리가 있으므로 optuna를 이용해서 확인  


## nei target prediction
traget값을 변경 count 에서 nei count로 변경  

> result notebook : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2023_01_02_TEST_new_data_training_spatial_aggregate_v2/result_veiw.ipynb


## double prediction

기존의 spatial agge 값을 추가한 target을 추가해서 기존의 target과 같이 예측도록 모델 변경  

```python
def get_training_mulit(data , max_prediction_length = 24 , max_encoder_length = 24*7):
    # traing data 생성
    training_cutoff = data["time_idx"].max() - max_prediction_length
    training = TimeSeriesDataSet(
        data[lambda x : x.time_idx <= training_cutoff],
        time_idx = "time_idx",
        target = ["count", 'nei_count'],
        group_ids = ['h_dong'],
        min_encoder_length=max_encoder_length // 2,  # keep encoder length long (as it is in the validation set)
        max_encoder_length=max_encoder_length,
        min_prediction_length=1,
        max_prediction_length=24,
        static_categoricals=["h_dong"],
        time_varying_known_categoricals=["HOD", "DOW" , 'isHoliday', 'MOY'],
        time_varying_known_reals=['pops'],
        time_varying_unknown_categoricals=['precip_form'],
        time_varying_unknown_reals=['count','windspd' , 'temp' ,'precip', 'nei1'], 
        target_normalizer=MultiNormalizer([
        GroupNormalizer(
            groups=["h_dong"], 
            transformation="relu",
            center = False
        ),
        GroupNormalizer(
            groups=["h_dong"], 
            transformation="softplus",
            center = False
        )]),
        add_relative_time_idx=True,
        add_target_scales=False,
        add_encoder_length=True,
        #allow_missing=True,
        allow_missing_timesteps = True,
        #predict_mode = False
        )
    return training
```
training에서 MultiNormalizer(GroupNormalizer() , GroupNormalizer())를 넣는 방식으로 변경  

```python
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.01,
    hidden_size=hidden_size,  
    attention_head_size=atten_head,
    dropout=0.1,
    hidden_continuous_size=hidden_size,
    output_size=[7,7],
    loss=QuantileLoss(),
    reduce_on_plateau_patience=4,
)
``` 
tft.output_size도 출력이 2개 이므로 list 형태로 2개의 출력하도록 설정  

각 함수마다 output harams를 추가해서 원하는 output prediction을 선택 할 수 있도록 변경  

> result notebook : http://202.31.200.194:8888/notebooks/NPLAB-NAS/Members/SEO/Emergency_Demand/TFT/2023_01_02_TEST_new_data_training_spatial_aggregate_v3/result_veiw.ipynb


## adjustment scaling

* aggeration을 했을 경우에는 원본의 scale 보다 크게 나올 수 있어서 이를 조정할 필요가 있어 보임    
* 기존의 target값을 사용했을 때도 rounding 값을 조정해야 할 수도 있음  
* 모델의 예측값을 보고 결정  

## Meeting
현재 이상태로 TFT 쪽은 나두고 refence model(LSTM, XGBoost) 모델 확인 및 train 진행  